# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices Exploration with `mlcroissant`
This notebook provides a complete guide for loading, exploring, and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load all metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print dataset summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"Version: {meta.version}")
print(f"Identifier: {meta.identifier}")

## 2. Data Overview
Review available **record sets** and **fields** in the dataset. All entities are referenced by their `@id`.

In [ ]:
# Display all RecordSet @ids with their field @ids

record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets defined in the Croissant metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id} (name='{field.name}')")
        print()

## 3. Data Extraction
Load data from each record set into a Pandas DataFrame for exploration. Use record set and field `@id` values as shown above.

In [ ]:
# Convert all RecordSet data into DataFrames, indexed by their @id

dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

if not record_set_ids:
    print("No record sets found for extraction.")
else:
    for record_set_id in record_set_ids:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for RecordSet {record_set_id}.")
        else:
            print(f"RecordSet {record_set_id} contained no records.")
        if record_set_id in dataframes:
            print(f"Columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")
            display(dataframes[record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply simple EDA steps such as numerical filtering, normalization, and aggregation, referencing fields via their `@id`.

If the main RecordSet contains numeric fields, we demonstrate the process below.

In [ ]:
# EDA: Select a numeric field by its @id
# Replace these @ids with those specific to your dataset from the Data Overview step.

# Example RecordSet and Field @ids -- update these after running previous cells!
example_record_set_id = None
numeric_field_id = None
group_field_id = None

# If record sets exist, select the first as an example
if record_set_ids:
    example_record_set_id = record_set_ids[0]
    df = dataframes[example_record_set_id]
    # Search for a likely numeric field
    for col in df.columns:
        if df[col].dtype in [float, int, 'float64', 'int64'] or all(pd.to_numeric(df[col], errors='coerce').notnull()):
            numeric_field_id = col
            break
    # Try to guess a group field
    common_group_fields = [c for c in df.columns if ('category' in c.lower() or 'group' in c.lower() or 'ward' in c.lower() or 'county' in c.lower())]
    if common_group_fields:
        group_field_id = common_group_fields[0]

if not example_record_set_id or not numeric_field_id:
    print("Could not automatically detect a numeric field; please inspect your DataFrame columns and select a suitable field manually.")
else:
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records in RecordSet '{example_record_set_id}' with '{numeric_field_id}' > {threshold:.2f}:")
    display(filtered_df.head())
    # Normalize numeric field
    normed_col = numeric_field_id + '_normalized'
    filtered_df[normed_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, normed_col]].head())
    # Grouping
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped statistics by '{group_field_id}':")
        display(grouped_df.head())

## 5. Visualization
Visualize the numeric field distribution, and relationships between fields. Update field `@ids` as needed based on your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram and boxplot of the normalized numeric field
if example_record_set_id and numeric_field_id:
    plt.figure(figsize=(10,4))
    sns.histplot(filtered_df[normed_col], bins=20, kde=True)
    plt.title(f"Distribution of normalized '{numeric_field_id}'")
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.show()
    
    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[normed_col])
        plt.title(f"Normalized '{numeric_field_id}' by '{group_field_id}'")
        plt.xlabel(group_field_id)
        plt.ylabel(f"{numeric_field_id} (normalized)")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No suitable numeric field for visualization found. Update the field @id variables as needed.")

## 6. Conclusion
In this notebook, we demonstrated how to load, review, and analyze record sets from the FAIR² dataset using `mlcroissant`. All entities are referenced by their `@id`, ensuring reproducibility and metadata clarity.

- We loaded dataset metadata and explored the content structure (record sets and fields).
- Using the discovered `@id`s, we loaded records into DataFrames and performed example exploratory data analysis (EDA), including outlier filtering and normalization.
- Finally, we visualized key numeric features for a deeper understanding of their distribution and group variation.

You can now further explore the dataset by analyzing other fields or customizing EDA steps for your research/analysis task.

**Remember:** Always use entity `@id` references when working with Croissant/FAIR datasets for clarity and future automation.